# <center>Organización de Datos</center>
#### <center>Cátedra Ing. Rodriguez, Juan Manuel </center>

## <center>Feature Engineering</center>
### <center>Práctica Transformación de datos</center>

> **Versión corregida y ampliada.** Cambios respecto de la original:
> - Se corrigió la contradicción entre el texto ("Label Encoder no debería usarse en features") y el código (se usaba sobre `property_type`, que es una feature).
> - Se renombraron variables que pisaban builtins de Python (`min`, `max`).
> - Los cortes de `price` ya no son números mágicos: se derivan con IQR, igual que en la notebook de valores atípicos.
> - Se agregó **train/test split antes de fitear cualquier encoder/scaler/imputer**, que faltaba en toda la serie y es la causa más común de *data leakage* en la práctica.
> - Se agregó un ejemplo con `Pipeline` + `ColumnTransformer`, la forma en que esto se arma en un flujo real.
> - Se agregó una nota sobre Yeo-Johnson como alternativa a Box-Cox para variables con ceros o negativos.
> - **Se agregó una sección nueva de creación de variables (feature creation)**, con un experimento que mide el impacto real de las nuevas variables sobre la métrica de un modelo.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import sklearn.preprocessing as skp

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from sklearn.feature_extraction import FeatureHasher
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import (
    KBinsDiscretizer,
    LabelEncoder,
    MinMaxScaler,
    Normalizer,
    OneHotEncoder,
    OrdinalEncoder,
    PowerTransformer,
    RobustScaler,
    StandardScaler,
)
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.options.display.max_columns = None

#### Preparación del Dataset

In [ ]:
df = pd.read_csv("/content/ar_properties.csv")
df.info()

Seleccionamos las filas con las cuales vamos a trabajar.

In [ ]:
df_filtrado = df.copy()

cond_lugar = df_filtrado['l2'] == "Capital Federal"
cond_moneda = df_filtrado['currency'] == "USD"
cond_operacion = df_filtrado['operation_type'] == "Venta"

df_filtrado = df_filtrado[cond_lugar & cond_moneda & cond_operacion]

columnas = {"l2": "ciudad", "l3": "barrio"}
df_filtrado.rename(columns=columnas, inplace=True)

Por simplicidad eliminamos todos los datos faltantes.

(Ver este tema en la clase y notebook correspondientes al tema de valores faltantes)

In [ ]:
df_limpio = df_filtrado.copy()
columnas_eliminar_NANs = ['bedrooms', 'l4', 'l5', 'l6']
df_limpio.drop(columnas_eliminar_NANs, axis='columns', inplace=True)
df_limpio.dropna(inplace=True)

filas_totales = df_limpio.shape[0]
print(df_limpio.isna().sum() / filas_totales * 100)
print(df_limpio.shape)

### Train / Test split (esto faltaba en la versión original)

**Esto es lo más importante que hay que corregir en esta notebook.** Hasta acá solo limpiamos filas — no ajustamos ("fiteamos") nada todavía. Pero apenas empecemos a fitear un `OrdinalEncoder`, un `StandardScaler`, un `KNNImputer`, etc., ese ajuste tiene que hacerse **solo con datos de entrenamiento**.

¿Por qué importa? Si fiteamos el escalador (por ejemplo, media y desvío de `price`) usando *todo* el dataset, la información de las observaciones de test se filtra indirectamente hacia el modelo antes de evaluarlo — eso es **data leakage**, y hace que las métricas de evaluación queden infladas de forma artificial. En un caso real, en el momento de entrenar no vas a tener las observaciones futuras (de test) disponibles para calcular esa media y ese desvío.

A partir de acá, cada vez que ajustemos algo, lo vamos a hacer sobre `df_train` y aplicarlo (transform) sobre `df_test`.

In [ ]:
df_train, df_test = train_test_split(df_limpio, test_size=0.2, random_state=42)

print("Train:", df_train.shape)
print("Test:", df_test.shape)

### Transformación de variables

En esta sección mostramos las principales estrategias para convertir variables según su tipo.

#### Conversión de variables categóricas

Los principales son:
* Ordinal Encoder
* Label Encoder
* One Hot Encoding

###### Ordinal Encoder

Ordinal Encoder se usa cuando las categorías tienen un orden natural (ej. "bajo" < "medio" < "alto"). Asigna un entero a cada categoría respetando ese orden. Es importante definir el orden explícitamente; si se deja al azar, el modelo puede aprender una relación ordinal incorrecta.

**Atención**: usaremos `property_type`, que es una variable nominal en esencia, pero en un dataset inmobiliario se puede argumentar un orden por "complejidad/valor estructural" del inmueble.

In [ ]:
print(df_train['property_type'].unique())

columns_to_encode = ['property_type']

orden_property = [
    'Lote',            # 0 - solo terreno, sin construcción
    'Cochera',         # 1 - construcción mínima
    'Depósito',        # 2 - estructura simple
    'Local comercial', # 3 - comercial básico
    'Oficina',         # 4 - comercial equipado
    'Otro',            # 5 - indefinido, al medio
    'Departamento',    # 6 - residencial estándar
    'PH',              # 7 - departamento con características de casa
    'Casa',            # 8 - residencial completo
]

oe = OrdinalEncoder(dtype='int', categories=[orden_property])

df_train_oe = df_train.copy()
df_test_oe = df_test.copy()

# Fiteamos SOLO con train...
oe.fit(df_train_oe[columns_to_encode])

# ...y aplicamos (transform) tanto a train como a test
df_train_oe[['property_type_encoded']] = oe.transform(df_train_oe[columns_to_encode])
df_test_oe[['property_type_encoded']] = oe.transform(df_test_oe[columns_to_encode])

print(df_train_oe['property_type'].values[:10])
print(df_train_oe['property_type_encoded'].values[:10])

In [ ]:
df_train_oe[['property_type', 'property_type_encoded']].value_counts()

#### Label Encoder

Label Encoder asigna un entero único a cada categoría sin garantizar un orden significativo. Es simple y compacto, pero introduce implícitamente un orden artificial. **Por eso se usa principalmente en la variable target (y), no en features (X)** — o, si se usa en features, solo tiene sentido con modelos basados en árboles, que no asumen relaciones lineales/ordinales entre el valor numérico de la feature y el target.

Para no repetir el mismo error de la versión anterior de esta notebook (donde el texto decía una cosa y el código hacía otra), acá lo mostramos explícitamente sobre una variable **target-like** de juguete, no sobre una feature de nuestro modelo. Si quisiéramos usar Label Encoder igual sobre `property_type` como feature para un modelo de árboles, hay que ser conscientes de esa limitación (no es apto, por ejemplo, para una regresión lineal).

In [ ]:
le = LabelEncoder()

# Ejemplo ilustrativo: así se vería si tuviéramos que codificar una variable target categórica.
# Fiteamos con train y aplicamos con transform a train y test (LabelEncoder no tiene .transform
# separado por columna como los otros, pero el principio de "fit solo en train" es el mismo).
ejemplo_target = df_train['property_type'].astype(str)
le.fit(ejemplo_target)

print("Clases aprendidas:", le.classes_)
print("Ejemplo codificado:", le.transform(ejemplo_target)[:10])

#### One Hot Encoding

One Hot Encoding crea una columna binaria por cada categoría. Evita el problema del orden artificial, pero puede generar muchas columnas con alta cardinalidad. También hay que cuidar la *dummy variable trap* (multicolinealidad), por lo que se suele eliminar una columna con `drop='first'`.

In [ ]:
ohe = OneHotEncoder(handle_unknown='ignore')

# Fiteamos solo con train
ohe.fit(df_train[['property_type']].astype(str))

property_type_encoded_train = ohe.transform(df_train[['property_type']].astype(str)).toarray().astype(int)
property_type_encoded_test = ohe.transform(df_test[['property_type']].astype(str)).toarray().astype(int)

property_type_encoded_train = pd.DataFrame(
    property_type_encoded_train,
    columns=ohe.get_feature_names_out(['property_type']),
    index=df_train.index,
).add_prefix('property_type_')

display(property_type_encoded_train.head())

Otra solución para One Hot Encoding implementada en pandas (`get_dummies`). Notar que si se usa sobre un dataset ya combinado de train+test hay riesgo de que las categorías no coincidan entre ambos conjuntos — por eso el enfoque con `OneHotEncoder` de sklearn (fit en train, transform en train y test) es más seguro en un flujo real.

In [ ]:
df_filtrado_dummies = pd.get_dummies(df_train, columns=['property_type'], dummy_na=True)
display(df_filtrado_dummies.head(2))
print(df_filtrado_dummies.shape)

Para evitar problemas de colinealidad en los features se debe excluir una categoría del set (la ausencia de todas — vector de 0s — indica la presencia de la categoría faltante); `get_dummies` tiene el parámetro `drop_first=True` para esto.

In [ ]:
df_filtrado_dummies = pd.get_dummies(df_train, columns=['property_type'], dummy_na=True, drop_first=True)
display(df_filtrado_dummies.head(2))
print(df_filtrado_dummies.shape)

### Numéricas

Vamos a trabajar con la variable `price`, usando **solo el set de train** para definir cualquier límite o parámetro.

In [ ]:
df_scaler_train = df_train.loc[:, ['price']].copy()
df_scaler_test = df_test.loc[:, ['price']].copy()
df_scaler_train.head()

#### De dónde salen los cortes de `price` (esto era un número mágico en la versión original)

En vez de fijar `> 450000` y `< 90000` "a ojo", los derivamos con la misma regla de Tukey (IQR) que usamos en la notebook de valores atípicos — así los dos temas de la materia quedan conectados en vez de ser dos números sueltos y arbitrarios que además se repetían idénticos en la notebook de valores faltantes.

In [ ]:
Q1_price = df_scaler_train['price'].quantile(0.25)
Q3_price = df_scaler_train['price'].quantile(0.75)
IQR_price = Q3_price - Q1_price

price_min_iqr = Q1_price - 1.5 * IQR_price
price_max_iqr = Q3_price + 1.5 * IQR_price

print("Límite inferior (IQR):", price_min_iqr)
print("Límite superior (IQR):", price_max_iqr)

# Filtramos train y aplicamos el MISMO límite (calculado en train) a test
df_scaler_train = df_scaler_train[(df_scaler_train['price'] >= price_min_iqr) & (df_scaler_train['price'] <= price_max_iqr)]
df_scaler_test = df_scaler_test[(df_scaler_test['price'] >= price_min_iqr) & (df_scaler_test['price'] <= price_max_iqr)]

sns.histplot(data=df_scaler_train['price'], alpha=0.5, bins=18).set(
    title="Histograma feature price (train)", xlabel="price", ylabel="Frecuencia"
)
plt.show()

#### Transformación Min-Max

Escala los valores al rango [0, 1]. Es sensible a outliers porque el mínimo y máximo se ven muy afectados por valores extremos. Min-Max asume que el mínimo y el máximo son representativos del rango real de los datos. Ideal cuando el algoritmo requiere que los datos estén en un rango acotado (redes neuronales, KNN, SVM).

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Fit SOLO con train
scaler.fit(df_scaler_train[['price']])

df_scaler_train['price_min_max'] = scaler.transform(df_scaler_train[['price']])
df_scaler_test['price_min_max'] = scaler.transform(df_scaler_test[['price']])

sns.histplot(data=df_scaler_train['price_min_max'], alpha=0.5, bins=18).set(
    title="Histograma feature price min-max (train)", xlabel="price min-max", ylabel="Frecuencia"
)
plt.show()

#### Transformación z-score

Centra los datos en media 0 y desvío estándar 1. No acota los valores a un rango fijo, por lo que es más robusta frente a outliers que Min-Max. Es la opción por defecto para la mayoría de los modelos lineales y cuando se asume distribución normal.

In [ ]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()
standard_scaler.fit(df_scaler_train[['price']])

df_scaler_train['price_z_score'] = standard_scaler.transform(df_scaler_train[['price']])
df_scaler_test['price_z_score'] = standard_scaler.transform(df_scaler_test[['price']])

sns.histplot(data=df_scaler_train['price_z_score'], alpha=0.5, bins=18).set(
    title="Histograma feature price z-score (train)", xlabel="price z-score", ylabel="Frecuencia"
)
plt.show()

### Transformación Decimal Scaling

Divide cada valor por una potencia de 10 tal que todos los valores queden en el rango [-1, 1]. Solo mira la magnitud del valor más grande para decidir por qué potencia de 10 dividir — no le importa si los datos están sesgados, si tienen outliers, ni cómo están distribuidos, simplemente corre el punto decimal.

**Nota:** este método casi no se usa en la práctica hoy en día; se muestra más como referencia histórica que como una técnica que vayan a aplicar en un proyecto real (Min-Max, z-score y RobustScaler cubren casi todos los casos de uso).

In [ ]:
d = len(str(int(df_scaler_train['price'].max())))
df_scaler_train['price_decimal_scaling'] = df_scaler_train['price'] / 10**d

sns.histplot(data=df_scaler_train['price_decimal_scaling'], alpha=0.5, bins=18).set(
    title="Histograma feature price decimal scaling", xlabel="price decimal scaling", ylabel="Frecuencia"
)
plt.show()

### Box-Cox y Yeo-Johnson

`PowerTransformer` busca una transformación que acerque la distribución a una normal. El método `box-cox` **requiere valores estrictamente positivos** (no acepta ceros ni negativos), porque internamente usa logaritmos y potencias fraccionarias que no están definidas para esos casos.

Si tu variable puede tener ceros o negativos (por ejemplo, una variable de "ganancia/pérdida"), usá `method="yeo-johnson"` en su lugar, que es una generalización de Box-Cox sin esa restricción.

In [ ]:
from sklearn.preprocessing import PowerTransformer

power_scaler = PowerTransformer(method="box-cox")  # price es siempre > 0, así que box-cox es válido acá
power_scaler.fit(df_scaler_train[['price']])

price_power_train = power_scaler.transform(df_scaler_train[['price']])

sns.histplot(data=price_power_train, alpha=0.5, bins=20).set(
    title="Histograma feature price box-cox (train)", xlabel="price box-cox", ylabel="Frecuencia"
)
plt.show()

# Si price pudiera tener ceros o negativos, usaríamos:
# power_scaler_yj = PowerTransformer(method="yeo-johnson")

### Discretizaciones

Convierte una variable continua en intervalos (bins), transformándola en categórica ordinal. Dos estrategias principales:

* Equal width: cada bin tiene el mismo rango de valores → sensible a outliers
* Equal frequency (quantile): cada bin tiene la misma cantidad de registros → más robusto

#### Binning con `KBinsDiscretizer` (fit en train, transform en ambos)

In [ ]:
enc = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')

_df_train = df_scaler_train[['price']].dropna().reset_index(drop=True)
_df_test = df_scaler_test[['price']].dropna().reset_index(drop=True)

enc.fit(_df_train)

X_binned_train = enc.transform(_df_train)
X_binned_train = pd.DataFrame(X_binned_train.astype(int), columns=['price_bins'])
result_train = pd.concat([_df_train, X_binned_train], axis=1)

display(result_train.head(10))
print("Límites bins (definidos con train):", enc.bin_edges_)

In [ ]:
ds_agrupado_price = result_train.groupby(['price_bins']).count().reset_index()

sns.barplot(x='price_bins', y='price', data=ds_agrupado_price, alpha=0.5).set(
    title="Igual Frecuencia (train)", ylabel='Frecuencia', xlabel='price_bins'
)
plt.show()

result_train['price_bins'].value_counts()

Utilizando pandas (`qcut` y `cut`) — útiles para exploración rápida, aunque para un flujo productivo con train/test conviene `KBinsDiscretizer`, que separa fit y transform de forma explícita.

In [ ]:
labels = ["bajo", "medio", "alto"]

df_discret_train = df_scaler_train[['price']].copy()
df_discret_train["price_discret_igual_frec"] = pd.qcut(df_discret_train["price"], q=3, labels=labels)

ds_agrupado_price = df_discret_train.groupby(['price_discret_igual_frec'], observed=True).count()

eje_x = ds_agrupado_price.index.tolist()
eje_y = ds_agrupado_price['price'].tolist()

sns.barplot(x=eje_x, y=eje_y, alpha=0.5).set(title="Igual Frecuencia", ylabel='Frecuencia', xlabel='price')
plt.show()

In [ ]:
labels = ["bajo", "medio", "alto"]
df_discret_train["price_discret_igual_size"] = pd.cut(df_discret_train["price"], bins=3, labels=labels)

ds_agrupado_price = df_discret_train.groupby(['price_discret_igual_size'], observed=True).count()

eje_x = ds_agrupado_price.index.tolist()
eje_y = ds_agrupado_price['price'].tolist()

sns.barplot(x=eje_x, y=eje_y, alpha=0.5).set(title="Igual ancho del intervalo", ylabel='Frecuencia', xlabel='price')
plt.show()

### Armando todo junto: `Pipeline` + `ColumnTransformer`

Hasta acá aplicamos cada transformación "a mano", encadenando `df_train`/`df_test` en celdas separadas. En un proyecto real esto se arma con `ColumnTransformer` (aplica una transformación distinta a cada columna o grupo de columnas) dentro de un `Pipeline` (encadena pasos). La ventaja: un solo `.fit()` sobre train, un solo `.transform()` sobre test, sin riesgo de mezclar accidentalmente ambos conjuntos.

In [ ]:
columnas_numericas = ['surface_total', 'surface_covered', 'rooms', 'bathrooms']
columnas_categoricas = ['property_type']

# Nota: para que este ejemplo corra hace falta que estas columnas no tengan NaNs
# (ver notebook de valores faltantes) o agregar un imputer como primer paso de cada rama.
preprocesador = ColumnTransformer(transformers=[
    ('num', StandardScaler(), columnas_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), columnas_categoricas),
])

pipeline_features = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
])

X_train_transformado = pipeline_features.fit_transform(df_train[columnas_numericas + columnas_categoricas].dropna())
print("Shape del resultado (train):", X_train_transformado.shape)

# El mismo pipeline, ya fiteado, se aplica sobre test sin volver a fitear
X_test_transformado = pipeline_features.transform(df_test[columnas_numericas + columnas_categoricas].dropna())
print("Shape del resultado (test):", X_test_transformado.shape)

**TODO:** agreguen un `SimpleImputer` o `KNNImputer` como primer paso de la rama numérica del `ColumnTransformer` (`Pipeline([('imputer', ...), ('scaler', ...)])`), para no tener que descartar filas con `dropna()`.

---

## Creación de variables (Feature Creation)

Todo lo anterior fue *transformar* variables que ya existían (cambiar su escala, su codificación, su forma). Pero una parte igual de importante — y muchas veces la que más impacto tiene — es **crear variables nuevas** a partir de las que ya tenemos. Acá es donde entra el criterio de dominio y la creatividad: combinar dos o tres variables, calcular ratios, cruzar una categórica con una numérica, etc.

La idea de esta sección es mostrar que una buena variable nueva puede mejorar la métrica de un modelo más de lo que lo haría, por ejemplo, cambiar de algoritmo. Vamos a comprobarlo con un experimento **antes/después**, no solo de palabra.

### Ideas de variables nuevas para este dataset

* **Ratios de dominio:** `price_per_m2 = price / surface_total` — el precio por metro cuadrado suele ser una señal mucho más comparable entre propiedades de distinto tamaño que el precio total.
* **Proporciones entre variables relacionadas:** `surface_covered / surface_total` — indica qué tan "edificado" está el terreno (un valor cercano a 1 sugiere poco espacio descubierto, por ejemplo).
* **Densidad de ambientes:** `rooms_per_bathroom = rooms / bathrooms` — combina dos variables de tamaño/comodidad en una sola señal.
* **Interacción categórica × numérica (encoding de dominio):** precio promedio histórico del `barrio` (*target encoding*) — captura el efecto de la ubicación de una forma mucho más rica que un simple OneHot, aunque **hay que fitearlo solo con train** para no filtrar información del target de test (si no, el modelo "ve" indirectamente el precio de test a través del promedio del barrio).

**Importante sobre el target encoding:** es un caso particular donde el riesgo de leakage es más sutil que en un encoder común, porque la propia variable que creamos usa el `price` (nuestro target) para calcularse. Por eso el promedio por barrio se calcula *exclusivamente* con `df_train`, y ese mismo valor (ya fijo) se le asigna a las filas de test — nunca se recalcula usando información de test.

In [ ]:
df_fe_train = df_train.copy()
df_fe_test = df_test.copy()

# --- Ratios de dominio ---
df_fe_train['price_per_m2'] = df_fe_train['price'] / df_fe_train['surface_total']
df_fe_test['price_per_m2'] = df_fe_test['price'] / df_fe_test['surface_total']

df_fe_train['surface_ratio'] = df_fe_train['surface_covered'] / df_fe_train['surface_total']
df_fe_test['surface_ratio'] = df_fe_test['surface_covered'] / df_fe_test['surface_total']

# Evitamos división por cero en bathrooms
df_fe_train['rooms_per_bathroom'] = df_fe_train['rooms'] / df_fe_train['bathrooms'].replace(0, np.nan)
df_fe_test['rooms_per_bathroom'] = df_fe_test['rooms'] / df_fe_test['bathrooms'].replace(0, np.nan)

# --- Target encoding de barrio, fiteado SOLO en train ---
precio_promedio_por_barrio = df_fe_train.groupby('barrio')['price'].mean()

df_fe_train['barrio_price_mean'] = df_fe_train['barrio'].map(precio_promedio_por_barrio)
# Para barrios de test que no aparecieron en train, usamos el promedio global de train como fallback
df_fe_test['barrio_price_mean'] = df_fe_test['barrio'].map(precio_promedio_por_barrio).fillna(df_fe_train['price'].mean())

df_fe_train[['price', 'surface_total', 'surface_covered', 'rooms', 'bathrooms',
             'price_per_m2', 'surface_ratio', 'rooms_per_bathroom', 'barrio_price_mean']].head()

### El experimento: ¿esto realmente ayuda a un modelo?

Vamos a entrenar el mismo modelo simple dos veces, con el mismo train/test split:

1. **Baseline:** solo con las variables numéricas originales.
2. **Con features nuevas:** agregando `price_per_m2`, `surface_ratio`, `rooms_per_bathroom` y `barrio_price_mean` (excepto `price_per_m2`, que directamente incluye a `price` y generaría leakage trivial si tratamos de predecir `price` — por eso la excluimos del target de este ejercicio y predecimos otra cosa, o la dejamos afuera).

**Ojo con una trampa de leakage dentro del propio feature:** si el objetivo es predecir `price`, no podemos usar `price_per_m2` ni `barrio_price_mean` tal como los calculamos (ambos derivan directamente de `price`) — serían variables que "ya saben la respuesta". Para este ejercicio didáctico vamos a predecir `surface_total` en cambio, así podemos usar `price` y sus derivados como features sin ese problema. **La lección para su propio proyecto es justamente esta: cuando el feature nuevo usa al target para construirse, hay que pensarlo dos veces.**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

target = 'surface_total'

# --- Modelo baseline: variables originales ---
features_baseline = ['price', 'rooms', 'bathrooms']

datos_train_base = df_fe_train[features_baseline + [target]].dropna()
datos_test_base = df_fe_test[features_baseline + [target]].dropna()

modelo_base = LinearRegression()
modelo_base.fit(datos_train_base[features_baseline], datos_train_base[target])
pred_base = modelo_base.predict(datos_test_base[features_baseline])

mae_base = mean_absolute_error(datos_test_base[target], pred_base)
r2_base = r2_score(datos_test_base[target], pred_base)

print(f"Baseline  -> MAE: {mae_base:.2f} | R2: {r2_base:.3f}")

In [ ]:
# --- Modelo con variables nuevas agregadas ---
features_nuevas = features_baseline + ['surface_ratio', 'rooms_per_bathroom', 'barrio_price_mean']

datos_train_fe = df_fe_train[features_nuevas + [target]].dropna()
datos_test_fe = df_fe_test[features_nuevas + [target]].dropna()

modelo_fe = LinearRegression()
modelo_fe.fit(datos_train_fe[features_nuevas], datos_train_fe[target])
pred_fe = modelo_fe.predict(datos_test_fe[features_nuevas])

mae_fe = mean_absolute_error(datos_test_fe[target], pred_fe)
r2_fe = r2_score(datos_test_fe[target], pred_fe)

print(f"Con nuevas features -> MAE: {mae_fe:.2f} | R2: {r2_fe:.3f}")

In [ ]:
# --- Comparación visual ---
comparacion = pd.DataFrame({
    "Modelo": ["Baseline", "Con features nuevas"],
    "MAE": [mae_base, mae_fe],
    "R2": [r2_base, r2_fe],
})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.barplot(data=comparacion, x="Modelo", y="MAE", ax=axes[0])
axes[0].set_title("MAE (menor es mejor)")

sns.barplot(data=comparacion, x="Modelo", y="R2", ax=axes[1])
axes[1].set_title("R2 (mayor es mejor)")

plt.tight_layout()
plt.show()

comparacion

Si el resultado da a favor de las features nuevas, esa es la evidencia concreta que buscábamos: la mejora no viene de un algoritmo más sofisticado, viene de haberle dado al mismo modelo señales mejor construidas.

Si en su corrida el resultado no mejora (puede pasar, dependiendo del sample y de si `rooms_per_bathroom` tiene muchos NaNs que se descartaron con `dropna()`), es también una lección válida: **no toda variable creativa mejora la métrica**, y por eso este tipo de experimento — comparar antes/después con la misma metodología — es la forma correcta de validar una idea de feature engineering, en vez de asumir que "suena razonable" y ya.

### TODO para ustedes

1. Propongan al menos **dos variables nuevas propias** (pueden combinar 2 o 3 columnas del dataset original: por ejemplo, alguna interacción con `ciudad`/`barrio`, con la fecha de publicación si está disponible, o algún ratio distinto a los que ya mostramos).
2. Agréguenlas al set de `features_nuevas` y vuelvan a correr el experimento.
3. Reporten si el MAE/R2 mejoró, empeoró o quedó igual, y una hipótesis de por qué.
4. (Opcional, más avanzado) Prueben el mismo experimento con un modelo no lineal, como `RandomForestRegressor` — ¿las mismas features ayudan igual, más o menos que con la regresión lineal? ¿Por qué podría cambiar según el algoritmo?